# 07 — XGBoost vs LightGBM for Tabular Competitions (Solutions)

## Learning Objectives
- Understand **why** each mathematical term appears and how it changes optimization/generalization.
- Apply the technique to both classification and regression.
- Compare synthetic and real-data behavior with reproducible experiments.

## Driving Question
> When should we prefer XGBoost vs LightGBM under real Kaggle constraints?

## Notebook Roadmap
Math (LaTeX) -> Synthetic Data -> Real Data (MNIST/California Housing) -> Visualizations -> Best Practices -> Exercises

### Mathematical Foundation (First Principles)

Both methods build additive tree ensembles:

$$
\hat{y}^{(t)}(x)=\hat{y}^{(t-1)}(x)+\eta f_t(x),\qquad
f_t \in \mathcal{F}_{\text{trees}}
$$

XGBoost objective (second-order approximation):

$$
\mathcal{L}^{(t)} \approx \sum_i \left[g_i f_t(x_i) + \frac{1}{2}h_i f_t(x_i)^2\right] + \Omega(f_t)
$$

LightGBM keeps boosting objective but changes tree-growth/search strategy (leaf-wise growth + histogram bins + GOSS/EFB) for faster optimization.

**Trade-off intuition**
- XGBoost: conservative/stable defaults, strong regularization controls.
- LightGBM: often faster on large/high-dimensional tables, but can overfit if leaf complexity is unchecked.

### Deep Equation Lineage (Term-by-Term)

We now connect every lesson to the same first-principles optimization pipeline.

1. **Population objective** (what we really care about):
$$
\mathcal{R}(\theta)=\mathbb{E}_{(x,y)\sim\mathcal{D}}[\ell(f_\theta(x),y)]
$$
2. **Empirical approximation** (what we can compute):
$$
\hat{\mathcal{R}}(\theta)=\frac{1}{N}\sum_{i=1}^{N}\ell(f_\theta(x_i),y_i)
$$
3. **Mini-batch stochastic estimate** (what we optimize per step):
$$
g_t=\frac{1}{m}\sum_{i \in \mathcal{B}_t}\nabla_\theta \ell(f_\theta(x_i),y_i)
$$
4. **Chain-rule expansion** (how gradients flow through layers):
$$
\nabla_\theta \ell
=
\frac{\partial \ell}{\partial \hat{y}}
\cdot
\frac{\partial \hat{y}}{\partial h_L}
\cdot
\frac{\partial h_L}{\partial h_{L-1}}
\cdots
\frac{\partial h_1}{\partial \theta}
$$
5. **Regularized update** (how each lesson perturbs the step):
$$
\theta_{t+1}
=
\theta_t
-\eta\left(g_t+\nabla_\theta\Omega(\theta_t)\right)
$$

| Term | Meaning | Where it appears in code |
|---|---|---|
| $\eta$ | step size controlling update magnitude | optimizer learning rate |
| $g_t$ | stochastic gradient estimate | `loss.backward()` |
| $\Omega(\theta)$ | explicit/implicit regularization | weight decay, dropout, early stopping |
| $m$ | mini-batch size | `DataLoader(..., batch_size=...)` |
| $\hat{y}=f_\theta(x)$ | model predictions | `logits = model(xb)` |

**Interpretation checkpoint:** if training is unstable, inspect whether the issue comes from gradient scale ($g_t$), step size ($\eta$), or noisy estimation (small $m$).

### Equation-to-Code Bridge (Detailed)

For each equation term, we map it directly to code objects:
- Objective terms -> `criterion(...)` and explicit regularization additions.
- Optimization step -> `loss.backward()` + `optimizer.step()`.
- Technique-specific parameters -> module flags (`BatchNorm1d`, `Dropout`, initialization hooks, patience logic, Optuna trial params).

In solution cells below, each advanced operation is annotated with **why** it exists.

### Before the Code: Purpose + Mechanics

This setup cell builds a reproducible tabular-competition environment with optional XGBoost/LightGBM support.
Background: robust benchmarking requires stable folds, consistent preprocessing, and graceful fallbacks when some libraries are unavailable.

In [ ]:
import copy
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.datasets import fetch_california_housing, make_classification
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, train_test_split, cross_validate
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import roc_auc_score, mean_squared_error, roc_curve
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

# Optional libraries: if unavailable, notebook still runs with sklearn fallbacks.
try:
    from xgboost import XGBRegressor, XGBClassifier
    XGB_AVAILABLE = True
except Exception:
    XGB_AVAILABLE = False

try:
    from lightgbm import LGBMRegressor, LGBMClassifier
    LGBM_AVAILABLE = True
except Exception:
    LGBM_AVAILABLE = False

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


def make_ohe():
    # Compatibility helper for older/newer sklearn versions.
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)


def build_tabular_frame():
    # Use California Housing and add competition-style categorical/missingness structure.
    raw = fetch_california_housing(as_frame=True)
    X = raw.frame.drop(columns=['MedHouseVal']).copy()
    y = raw.frame['MedHouseVal'].astype(float).copy()

    X['income_bucket'] = pd.qcut(X['MedInc'], q=5, labels=False, duplicates='drop').astype(str)
    X['house_age_bucket'] = pd.cut(X['HouseAge'], bins=[0, 15, 30, 45, 60], include_lowest=True).astype(str)
    X['geo_bucket'] = (X['Latitude'].round(1).astype(str) + '_' + X['Longitude'].round(1).astype(str))

    # Inject mild missingness to mimic real competition cleanup requirements.
    missing_mask = np.random.rand(len(X)) < 0.03
    X.loc[missing_mask, 'AveRooms'] = np.nan
    X.loc[missing_mask, 'AveBedrms'] = np.nan
    return X, y


class TabularMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=(256, 128), dropout_p=0.15, output_dim=1):
        super().__init__()
        layers = []
        prev = input_dim
        for hidden in hidden_dims:
            layers.append(nn.Linear(prev, hidden))
            layers.append(nn.BatchNorm1d(hidden))
            layers.append(nn.ReLU())
            if dropout_p > 0:
                layers.append(nn.Dropout(dropout_p))
            prev = hidden
        layers.append(nn.Linear(prev, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def make_tabular_preprocessor(num_cols, cat_cols):
    return ColumnTransformer(
        transformers=[
            ('num', Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())
            ]), num_cols),
            ('cat', Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('ohe', make_ohe())
            ]), cat_cols),
        ]
    )


def fit_preprocessor_matrices(preprocessor, X_train, X_valid):
    X_train_m = preprocessor.fit_transform(X_train)
    X_valid_m = preprocessor.transform(X_valid)
    if hasattr(X_train_m, 'toarray'):
        X_train_m = X_train_m.toarray()
    if hasattr(X_valid_m, 'toarray'):
        X_valid_m = X_valid_m.toarray()
    return np.asarray(X_train_m, dtype=np.float32), np.asarray(X_valid_m, dtype=np.float32)


def make_tabular_loaders(X_train_np, y_train, X_valid_np, y_valid, batch_size=256, task='regression'):
    y_train_t = torch.tensor(np.asarray(y_train, dtype=np.float32).reshape(-1, 1), dtype=torch.float32)
    y_valid_t = torch.tensor(np.asarray(y_valid, dtype=np.float32).reshape(-1, 1), dtype=torch.float32)
    train_ds = TensorDataset(torch.tensor(X_train_np, dtype=torch.float32), y_train_t)
    valid_ds = TensorDataset(torch.tensor(X_valid_np, dtype=torch.float32), y_valid_t)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    valid_loader = DataLoader(valid_ds, batch_size=batch_size, shuffle=False)
    return train_loader, valid_loader


def train_tabular_mlp(
    X_train,
    y_train,
    X_valid,
    y_valid,
    task='regression',
    hidden_dims=(256, 128),
    dropout_p=0.15,
    lr=8e-4,
    weight_decay=1e-5,
    batch_size=256,
    epochs=40,
    patience=6,
    verbose=False,
):
    num_cols = X_train.select_dtypes(include=['number']).columns.tolist()
    cat_cols = [c for c in X_train.columns if c not in num_cols]
    preprocessor = make_tabular_preprocessor(num_cols, cat_cols)
    X_train_np, X_valid_np = fit_preprocessor_matrices(preprocessor, X_train, X_valid)
    train_loader, valid_loader = make_tabular_loaders(
        X_train_np, y_train, X_valid_np, y_valid, batch_size=batch_size, task=task
    )

    model = TabularMLP(
        input_dim=X_train_np.shape[1],
        hidden_dims=hidden_dims,
        dropout_p=dropout_p,
        output_dim=1,
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.BCEWithLogitsLoss() if task == 'classification' else nn.MSELoss()

    history = {'train_loss': [], 'val_loss': []}
    best_state = copy.deepcopy(model.state_dict())
    best_val = float('inf')
    best_epoch = 1
    wait = 0
    stopped_epoch = epochs

    for epoch in range(epochs):
        model.train()
        train_losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in valid_loader:
                xb, yb = xb.to(device), yb.to(device)
                out = model(xb)
                val_losses.append(criterion(out, yb).item())

        train_loss = float(np.mean(train_losses))
        val_loss = float(np.mean(val_losses))
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)

        if verbose:
            print(f'Epoch {epoch+1:02d}: train={train_loss:.5f} val={val_loss:.5f}')

        if val_loss < best_val - 1e-6:
            best_val = val_loss
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1
            wait = 0
        else:
            wait += 1

        if wait >= patience:
            stopped_epoch = epoch + 1
            break

    model.load_state_dict(best_state)

    with torch.no_grad():
        valid_tensor = torch.tensor(X_valid_np, dtype=torch.float32).to(device)
        valid_logits = model(valid_tensor).squeeze(1).cpu().numpy()

    if task == 'classification':
        valid_pred = 1.0 / (1.0 + np.exp(-valid_logits))
    else:
        valid_pred = valid_logits

    return {
        'model': model,
        'preprocessor': preprocessor,
        'history': history,
        'val_pred': valid_pred,
        'best_epoch': best_epoch,
        'stopped_epoch': stopped_epoch,
    }


def predict_tabular_mlp(bundle, X_df, task='regression'):
    X_np = bundle['preprocessor'].transform(X_df)
    if hasattr(X_np, 'toarray'):
        X_np = X_np.toarray()
    X_tensor = torch.tensor(np.asarray(X_np, dtype=np.float32), dtype=torch.float32).to(device)
    with torch.no_grad():
        logits = bundle['model'](X_tensor).squeeze(1).cpu().numpy()
    if task == 'classification':
        return 1.0 / (1.0 + np.exp(-logits))
    return logits


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


TABULAR_MLP_BASE_CFG = {
    'hidden_dims': (256, 128),
    'dropout_p': 0.15,
    'lr': 8e-4,
    'weight_decay': 1e-5,
    'batch_size': 256,
    'epochs': 40,
    'patience': 6,
}


def run_tabular_mlp_cv(X_df, y, cfg, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    oof_pred = np.zeros(len(X_df), dtype=np.float32)
    fold_scores = []

    for fold_idx, (tr_idx, va_idx) in enumerate(kf.split(X_df), start=1):
        X_tr, X_va = X_df.iloc[tr_idx], X_df.iloc[va_idx]
        y_tr, y_va = np.asarray(y)[tr_idx], np.asarray(y)[va_idx]
        bundle = train_tabular_mlp(X_tr, y_tr, X_va, y_va, task='regression', **cfg)
        pred = bundle['val_pred']
        score = rmse(y_va, pred)
        fold_scores.append(score)
        oof_pred[va_idx] = pred
        print(f'Fold {fold_idx}: RMSE={score:.5f}, best_epoch={bundle["best_epoch"]}, stop_epoch={bundle["stopped_epoch"]}')

    return {
        'fold_scores': fold_scores,
        'oof_pred': oof_pred,
        'rmse_mean': float(np.mean(fold_scores)),
        'rmse_std': float(np.std(fold_scores)),
    }

### After the Code: Background + Why It Can Help

This setup emphasizes reproducibility and fairness: same folds, same preprocessing skeleton, and explicit fallback behavior.
For lessons 08 and 09 we now share one PyTorch MLP pipeline (preprocessing -> loaders -> training loop -> inference) so modeling choices stay aligned.

## XGBoost vs LightGBM — Controlled Regression Benchmark
We compare both frameworks on the same folds, preprocessing, and metric to isolate algorithmic/runtime trade-offs.

### Before the Code: Purpose + Mechanics

This experiment performs a leakage-safe CV benchmark on a realistic tabular regression frame.
Background: comparing RMSE without runtime, variance, and model complexity can hide practical Kaggle iteration costs.

In [ ]:
X, y = build_tabular_frame()
numeric_cols = X.select_dtypes(include=['number']).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numeric_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ohe', make_ohe())
        ]), categorical_cols),
    ]
)

cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
rows = []
fold_rows = []

reg_models = {
    'xgboost': (
        XGBRegressor(
            n_estimators=450, learning_rate=0.04, max_depth=6,
            subsample=0.85, colsample_bytree=0.80,
            objective='reg:squarederror', random_state=SEED
        ) if XGB_AVAILABLE else HistGradientBoostingRegressor(
            learning_rate=0.04, max_depth=8, random_state=SEED
        )
    ),
    'lightgbm': (
        LGBMRegressor(
            n_estimators=450, learning_rate=0.04, num_leaves=31,
            subsample=0.85, colsample_bytree=0.80, random_state=SEED
        ) if LGBM_AVAILABLE else HistGradientBoostingRegressor(
            learning_rate=0.04, max_depth=12, random_state=SEED
        )
    ),
}

for name, model in reg_models.items():
    pipe = Pipeline([('prep', preprocessor), ('model', model)])
    t0 = time.perf_counter()
    cv_out = cross_validate(
        pipe, X, y,
        scoring='neg_root_mean_squared_error',
        cv=cv,
        n_jobs=-1,
        return_train_score=False
    )
    elapsed = time.perf_counter() - t0
    rows.append({
        'model': name,
        'cv_rmse_mean': -cv_out['test_score'].mean(),
        'cv_rmse_std': cv_out['test_score'].std(),
        'wall_clock_sec': elapsed
    })
    for fold_idx, fold_score in enumerate(cv_out['test_score'], start=1):
        fold_rows.append({'model': name, 'fold': fold_idx, 'rmse': -fold_score})

comparison_reg = pd.DataFrame(rows).sort_values('cv_rmse_mean')
fold_reg = pd.DataFrame(fold_rows)
display(comparison_reg)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.barplot(data=comparison_reg, x='model', y='cv_rmse_mean', ax=axes[0], palette='deep')
axes[0].errorbar(
    x=np.arange(len(comparison_reg)),
    y=comparison_reg['cv_rmse_mean'],
    yerr=comparison_reg['cv_rmse_std'],
    fmt='none',
    ecolor='black',
    capsize=4
)
axes[0].set_title('CV RMSE mean ± std')

sns.boxplot(data=fold_reg, x='model', y='rmse', ax=axes[1], palette='Set2')
sns.stripplot(data=fold_reg, x='model', y='rmse', ax=axes[1], color='black', alpha=0.55, size=4)
axes[1].set_title('Fold-level RMSE distribution')

sns.scatterplot(data=comparison_reg, x='wall_clock_sec', y='cv_rmse_mean', hue='model', s=130, ax=axes[2])
for _, row in comparison_reg.iterrows():
    axes[2].annotate(row['model'], (row['wall_clock_sec'], row['cv_rmse_mean']), xytext=(4, 4), textcoords='offset points')
axes[2].set_title('Score vs runtime frontier')
axes[2].set_xlabel('Wall clock (sec)')
axes[2].set_ylabel('CV RMSE mean')
plt.tight_layout()
plt.show()

### After the Code: Background + Why It Can Help

Interpretation checklist:
1. Lower mean RMSE matters, but standard deviation indicates fold stability.
2. Runtime per sweep affects how many ideas you can test before deadline.
3. If score differences are tiny, prefer the framework with cleaner iteration/debugging in your stack.

## XGBoost vs LightGBM — Classification Side Experiment
Kaggle datasets frequently use AUC/F1 objectives, so we run a second benchmark focused on ranking quality.

### Before the Code: Purpose + Mechanics

This side experiment compares both methods on synthetic imbalanced classification using ROC-AUC.
Background: ranking metrics are sensitive to calibration and split decisions, exposing different strengths than RMSE.

In [ ]:
Xc, yc = make_classification(
    n_samples=7000, n_features=28, n_informative=16, n_redundant=6,
    weights=[0.78, 0.22], class_sep=1.1, random_state=SEED
)
Xc = pd.DataFrame(Xc, columns=[f'f_{i:02d}' for i in range(Xc.shape[1])])
Xc['f_00_bucket'] = pd.qcut(Xc['f_00'], q=6, labels=False, duplicates='drop').astype(str)

Xc_tr, Xc_va, yc_tr, yc_va = train_test_split(
    Xc, yc, test_size=0.25, stratify=yc, random_state=SEED
)
num_cols = Xc_tr.select_dtypes(include=['number']).columns.tolist()
cat_cols = [c for c in Xc_tr.columns if c not in num_cols]

prep_cls = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), num_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ohe', make_ohe())
        ]), cat_cols),
    ]
)

cls_models = {
    'xgboost': (
        XGBClassifier(
            n_estimators=420, learning_rate=0.05, max_depth=5,
            subsample=0.85, colsample_bytree=0.85,
            eval_metric='auc', random_state=SEED
        ) if XGB_AVAILABLE else HistGradientBoostingClassifier(
            learning_rate=0.05, max_depth=8, random_state=SEED
        )
    ),
    'lightgbm': (
        LGBMClassifier(
            n_estimators=420, learning_rate=0.05, num_leaves=31,
            subsample=0.85, colsample_bytree=0.85, random_state=SEED
        ) if LGBM_AVAILABLE else LogisticRegression(max_iter=800)
    ),
}

rows_cls = []
roc_rows = []
proba_rows = []
for name, model in cls_models.items():
    pipe = Pipeline([('prep', prep_cls), ('model', model)])
    t0 = time.perf_counter()
    pipe.fit(Xc_tr, yc_tr)
    fit_time = time.perf_counter() - t0
    pred_proba = pipe.predict_proba(Xc_va)[:, 1]
    fpr, tpr, _ = roc_curve(yc_va, pred_proba)
    rows_cls.append({
        'model': name,
        'val_auc': roc_auc_score(yc_va, pred_proba),
        'fit_time_sec': fit_time
    })
    roc_rows.append({'model': name, 'fpr': fpr, 'tpr': tpr})
    proba_rows.append(pd.DataFrame({'model': name, 'pred_proba': pred_proba, 'target': yc_va}))

comparison_cls = pd.DataFrame(rows_cls).sort_values('val_auc', ascending=False)
proba_df = pd.concat(proba_rows, ignore_index=True)
display(comparison_cls)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].bar(comparison_cls['model'], comparison_cls['val_auc'], color=['tab:blue', 'tab:green'])
axes[0].set_title('Validation AUC comparison')
axes[0].set_ylim(max(0.5, comparison_cls['val_auc'].min() - 0.03), min(1.0, comparison_cls['val_auc'].max() + 0.03))
axes[1].bar(comparison_cls['model'], comparison_cls['fit_time_sec'], color=['tab:blue', 'tab:green'])
axes[1].set_title('Fit time (seconds)')
for row in roc_rows:
    axes[2].plot(row['fpr'], row['tpr'], label=row['model'])
axes[2].plot([0, 1], [0, 1], linestyle='--', color='gray', linewidth=1)
axes[2].set_title('ROC curves')
axes[2].set_xlabel('False positive rate')
axes[2].set_ylabel('True positive rate')
axes[2].legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
sns.kdeplot(data=proba_df, x='pred_proba', hue='model', common_norm=False, fill=True, alpha=0.25)
plt.title('Prediction probability density by model')
plt.xlabel('Predicted positive probability')
plt.tight_layout()
plt.show()

### After the Code: Background + Why It Can Help

Practical trade-offs to record in your experiment log:
- Which model gives better score per second?
- Which one is easier to tune under your compute budget?
- Which one appears less sensitive to fold-specific variance?

**Expected outcomes (typical):**
- Both frameworks are usually close on score; winner often flips by dataset/folds.
- LightGBM often trains faster on large sparse matrices; XGBoost can be slightly more stable with conservative defaults.
- If score deltas are < 0.003, prioritize reproducibility and iteration speed over tiny single-run gains.

### Before the Code: Purpose + Mechanics

This optional solved extension performs a compact hyperparameter sensitivity sweep for both frameworks.
Background: a small sweep can reveal whether your current winner is robust or a narrow local optimum.

In [ ]:
search_grid = [
    {'name': 'xgboost', 'lr': 0.03, 'depth_or_leaves': 5},
    {'name': 'xgboost', 'lr': 0.06, 'depth_or_leaves': 7},
    {'name': 'lightgbm', 'lr': 0.03, 'depth_or_leaves': 31},
    {'name': 'lightgbm', 'lr': 0.06, 'depth_or_leaves': 63},
]

quick_rows = []
for row in search_grid:
    if row['name'] == 'xgboost' and XGB_AVAILABLE:
        model = XGBRegressor(
            n_estimators=260, learning_rate=row['lr'], max_depth=row['depth_or_leaves'],
            subsample=0.85, colsample_bytree=0.8, objective='reg:squarederror', random_state=SEED
        )
    elif row['name'] == 'lightgbm' and LGBM_AVAILABLE:
        model = LGBMRegressor(
            n_estimators=260, learning_rate=row['lr'], num_leaves=row['depth_or_leaves'],
            subsample=0.85, colsample_bytree=0.8, random_state=SEED
        )
    else:
        model = HistGradientBoostingRegressor(learning_rate=row['lr'], max_depth=8, random_state=SEED)

    pipe = Pipeline([('prep', preprocessor), ('model', model)])
    scores = cross_validate(pipe, X, y, cv=cv, scoring='neg_root_mean_squared_error', n_jobs=-1)['test_score']
    quick_rows.append({**row, 'rmse_mean': -scores.mean(), 'rmse_std': scores.std()})

display(pd.DataFrame(quick_rows).sort_values('rmse_mean'))

### After the Code: Background + Why It Can Help

Use this table to decide whether to invest deeper tuning budget in one framework or maintain both in your ensemble pool.
The best production choice is often the model with strong mean score and low variance.

## Best Practices
- Tune learning rate and number of estimators jointly (small eta -> more trees).
- Use cross-validation with fixed folds for fair framework comparison.
- Track both score and wall-clock time; Kaggle iteration speed matters.

## Common Pitfalls
- Applying a technique without a baseline comparison.
- Drawing conclusions from training loss only.
- Ignoring seed sensitivity and run-to-run variance.

## Explanatory Depth Checkpoints
- **Why this workflow?** Because leaderboard gains are fragile without leakage-safe validation and reproducible logs.
- **Key idea:** Strong experiments isolate one hypothesis at a time so score movement has a clear causal explanation.
- **Important pitfall:** A public-score spike can hide private overfitting; always compare fold variance and shake risk.
- **In practice:** Keep assumptions explicit, test alternatives quickly, and record rollback plans before each submission.
- **Question:** Which diagnostic would make you reject a seemingly strong model?
- **Question:** How would you defend your final submission choice to a teammate under deadline pressure?

## Exercises (Competition-Oriented)
1. **Exercise:** Re-run all experiments with a second seed and quantify score/rank variance.
2. **Exercise:** Replace one preprocessing block and document exact before/after effects.
3. **Exercise:** Perform a 3-row ablation table: baseline, +feature/model change, +tuning change.
4. **Exercise:** Add one failure analysis plot and explain what action it suggests.
5. **Exercise:** Build a strict 30-minute local runtime pipeline and maximize validation score.
6. **Question:** Which feature block improved mean score but harmed fold stability, and why?
7. **Exercise:** Reproduce your best run with one different fold schema and compare ranking movement.
8. **Exercise:** Add a simple blend and report whether shake risk improved versus single-model baseline.
9. **Question:** If compute is cut by 50%, which experiments stay and which are dropped?
10. **Exercise:** Write a short decision memo defending one final submission and one rollback candidate.

### Solution Depth Notes
- Includes complete runnable code for realistic dataset-driven tasks.
- Reports expected metric behavior and variance, not just single-run numbers.
- Provides alternative path: CatBoost can outperform both when categorical handling and minimal preprocessing are priorities.

### Before the Code: Purpose + Mechanics

This solved scaffold demonstrates a disciplined experiment registry and decision protocol.
Background: strong Kaggle results come from repeatable process more than isolated tricks.

In [ ]:
experiment_log = pd.DataFrame([
    {'exp_id': 'base_001', 'feature_set': 'raw', 'model': 'hist_gbdt', 'cv_metric': 0.585, 'public_metric': 0.592, 'notes': 'baseline'},
    {'exp_id': 'feat_014', 'feature_set': 'raw+ratios+geo_oof', 'model': 'hist_gbdt', 'cv_metric': 0.561, 'public_metric': 0.566, 'notes': 'feature lift'},
    {'exp_id': 'blend_021', 'feature_set': 'feat_014', 'model': 'tree_blend', 'cv_metric': 0.554, 'public_metric': 0.558, 'notes': 'blend + shake control'},
])

experiment_log['delta_cv'] = experiment_log['cv_metric'] - experiment_log['cv_metric'].iloc[0]
experiment_log['delta_public'] = experiment_log['public_metric'] - experiment_log['public_metric'].iloc[0]
display(experiment_log)

decision_rule = '''
1) Filter experiments by reproducibility + leakage checks.
2) Select top candidates by CV mean and variance.
3) Break ties using simulated shake or private-proxy behavior.
4) Submit with explicit rollback candidate.
'''.strip()
print(decision_rule)

### After the Code: Background + Why It Can Help

Keep this structure in every project to make model selection auditable and stress-tested.
The key competency is reliable decision quality under uncertainty.